# Lab 5: Kafka — producent, konsument, JSON, klucze

## Cel

- Zaawansowane użycie producenta i konsumenta Kafka w Pythonie,
- Praca z kluczami wiadomości i partycjonowaniem,
- Przetwarzanie zdarzeń w czasie rzeczywistym,
- Połączenie Kafki z modelem ML.

---

## Część 1: Producent zdarzeń e-commerce

### Zadanie 1.1 — Generator zdarzeń

Napisz producenta, który symuluje zdarzenia ze sklepu internetowego.
Każde zdarzenie to JSON z polami:
- `event_id` (unikalny)
- `user_id` (losowy z u01–u50)
- `event_type` ("view", "add_to_cart", "purchase" z wagami 60%, 25%, 15%)
- `product_id` (p100–p120)
- `price` (10–1000 PLN)
- `timestamp` (aktualny czas ISO)

Klucz wiadomości: `user_id` (aby zdarzenia tego samego użytkownika trafiały do jednej partycji).

In [ ]:
%%file ecommerce_producer.py
from confluent_kafka import Producer
import json, random, time
from datetime import datetime

conf = {'bootstrap.servers': 'localhost:9092'}
producer = Producer(conf)

def generate_event():
    # TWÓJ KOD
    # Zwroc slownik z polami: event_id, user_id, event_type, product_id, price, timestamp
    pass

# TWÓJ KOD
# Wyslij 100 zdarzen do tematu 'ecommerce-events'
# Uzyj key=user_id.encode('utf-8')

### Zadanie 1.2 — Utwórz temat z odpowiednią liczbą partycji

```bash
docker compose exec kafka kafka-topics --create \
  --topic ecommerce-events \
  --bootstrap-server localhost:9092 \
  --partitions 5 \
  --replication-factor 1
```

---

## Część 2: Konsumenci specjalizowani

### Zadanie 2.1 — Konsument: filtr zakupów

Napisz konsumenta, który wyświetla tylko zdarzenia typu `purchase`.

In [ ]:
%%file purchase_consumer.py
from confluent_kafka import Consumer
import json

conf = {
    'bootstrap.servers': 'localhost:9092',
    'group.id': 'purchase-tracker',
    'auto.offset.reset': 'earliest'
}

# TWÓJ KOD
# Subskrybuj 'ecommerce-events'
# Filtruj i wyswietlaj tylko event_type == 'purchase'

### Zadanie 2.2 — Konsument: statystyki na żywo

Napisz konsumenta (inna `group.id`!), który na bieżąco liczy:
- liczbę zdarzeń per `event_type`
- sumę wartości zakupów (tylko purchase)
- top 3 najczęściej kupowane produkty

Wyświetlaj statystyki co 10 wiadomości.

In [ ]:
%%file stats_consumer.py
from confluent_kafka import Consumer
from collections import Counter, defaultdict
import json

# TWÓJ KOD

---

## Część 3: Kafka + Model ML

### Zadanie 3.1 — Konsument z detekcją anomalii

Wykorzystaj model z Lab 3 (`fraud_model.pkl`) lub prosty model oparty na regule:
- jeśli `price > 800` i `event_type == 'purchase'` → ALERT

Napisz konsumenta, który dla każdego zdarzenia `purchase` sprawdza, czy jest podejrzane.

In [ ]:
%%file fraud_consumer.py
from confluent_kafka import Consumer
import json
# import pickle  # jesli uzywasz modelu z Lab 3

# TWÓJ KOD
# Odczytuj zdarzenia z 'ecommerce-events'
# Dla kazdego purchase sprawdzaj regule lub model
# Wyswietlaj ALERT dla podejrzanych transakcji

### Zadanie 3.2 — Uruchom wszystko razem

W trzech terminalach uruchom:
1. `python ecommerce_producer.py`
2. `python purchase_consumer.py`
3. `python fraud_consumer.py`

Obserwuj, jak różni konsumenci przetwarzają ten sam strumień niezależnie.

---

## Praca domowa

1. Dodaj nowy temat `alerts`, do którego `fraud_consumer` wysyła wykryte anomalie (producent + konsument w jednym!).
2. Napisz konsumenta tematu `alerts`, który loguje alerty do pliku CSV.
3. Wyślij kod do repozytorium Git.

Na następnych zajęciach: Apache Spark — DataFrame i SQL.